In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import uuid
from scipy.stats import linregress

# Constants
MAX_DAYS = 365  # For DaysSinceLastPurchase for cold SKUs
ROLLING_WINDOWS = [4, 8, 12]  # Multiple rolling window sizes
NEGATIVE_SAMPLING_RATIO = 5  # Negative to positive sample ratio
NUM_WEEKS = 10  # Number of historical weeks for training
INFERENCE_WEEK = 11  # Week for inference

CURRENT_DATE = datetime.now()

In [2]:
# Generate synthetic data  
def generate_synthetic_data():
    skus = pd.DataFrame({
        'SKUID': [f'SKU{i}' for i in range(1, 101)],
        'Manufacturer': np.random.choice(['M1', 'M2', 'M3'], 100),
        'Category': np.random.choice(['Cat1', 'Cat2', 'Cat3'], 100),
        'Segment': np.random.choice(['Seg1', 'Seg2'], 100)
    })
    
    customers = pd.DataFrame({
        'CustomerID': [f'C{i}' for i in range(1, 51)],
        'State': np.random.choice(['State1', 'State2'], 50),
        'City': np.random.choice(['City1', 'City2'], 50),
        'Town': np.random.choice(['Town1', 'Town2'], 50),
        'Recency': np.random.randint(1, 100, 50),
        'Frequency': np.random.randint(1, 50, 50),
        'Monetary': np.random.uniform(100, 10000, 50)
    })
    
    transactions = []
    start_date = datetime(2025, 1, 1)
    for week in range(1, NUM_WEEKS + 1):
        week_date = start_date + timedelta(weeks=week - 1)
        for _ in range(200):
            customer = np.random.choice(customers['CustomerID'])
            sku = np.random.choice(skus['SKUID'])
            transactions.append({
                'Week': week,
                'CustomerID': customer,
                'SKUID': sku,
                'OrderValue': np.random.uniform(10, 500),
                'TransactionDate': week_date
            })
    transactions = pd.DataFrame(transactions)
    
    return skus, customers, transactions


In [ ]:
# Updated feature engineering function with multiple rolling windows
def compute_features(week, transactions, skus, customers, is_inference=False):
    if is_inference:
        hist_transactions = transactions[transactions['Week'] <= week - 1]
    else:
        hist_transactions = transactions[transactions['Week'] < week]
    
    features_list = []
    
    for _, row in transactions[transactions['Week'] == week].iterrows() if not is_inference else transactions.iterrows():
        customer_id = row['CustomerID'] if not is_inference else row['CustomerID']
        sku_id = row['SKUID'] if not is_inference else row['SKUID']
        feature_dict = {'Week': week, 'CustomerID': customer_id, 'SKUID': sku_id}
        
        sku_info = skus[skus['SKUID'] == sku_id].iloc[0] if sku_id in skus['SKUID'].values else None
        customer_info = customers[customers['CustomerID'] == customer_id].iloc[0]
        
        if sku_info is None:
            continue
        
        # Customer-SKU Interaction Features
        customer_sku_transactions = hist_transactions[
            (hist_transactions['CustomerID'] == customer_id) & 
            (hist_transactions['SKUID'] == sku_id)
        ]
        
        feature_dict['DaysSinceLastPurchase_SKU'] = (
            MAX_DAYS + 1 if customer_sku_transactions.empty 
            else (datetime(2025, 1, 1) + timedelta(weeks=week - 1) - 
                  customer_sku_transactions['TransactionDate'].max()).days
        )
        
        feature_dict['TotalPurchases_SKU'] = len(customer_sku_transactions)
        feature_dict['AvgOrderValue_SKU_by_Customer'] = (
            customer_sku_transactions['OrderValue'].mean() if not customer_sku_transactions.empty else 0
        )
        
        # Customer General Behavior (RFM)
        feature_dict['Customer_Recency'] = customer_info['Recency']
        feature_dict['Customer_Frequency'] = customer_info['Frequency']
        feature_dict['Customer_Monetary'] = customer_info['Monetary']
        feature_dict['Customer_TotalUniqueSKUsPurchased_Overall'] = len(
            hist_transactions[hist_transactions['CustomerID'] == customer_id]['SKUID'].unique()
        )
        
        # Localized SKU Performance (State)
        state = customer_info['State']
        state_transactions = hist_transactions[hist_transactions['CustomerID'].isin(
            customers[customers['State'] == state]['CustomerID']
        )]
        feature_dict['SKU_TotalSales_CustomerState_Overall'] = state_transactions[
            state_transactions['SKUID'] == sku_id
        ]['OrderValue'].sum()
        
        # Time-Series & Rolling Window (Multiple Windows: 4, 8, 12 weeks)
        for window in ROLLING_WINDOWS:
            rolling_transactions = hist_transactions[
                (hist_transactions['Week'] >= week - window) & 
                (hist_transactions['Week'] <= week - 1)
            ]
            customer_rolling = rolling_transactions[rolling_transactions['CustomerID'] == customer_id]
            
            # Rolling features for each window
            feature_dict[f'Customer_RollingAvgOrderValue_{window}Weeks'] = (
                customer_rolling['OrderValue'].mean() if not customer_rolling.empty else 0
            )
            feature_dict[f'Customer_RollingUniqueSKUsPurchased_{window}Weeks'] = len(customer_rolling['SKUID'].unique())
            feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] = len(customer_rolling)
            
            # SKU-specific rolling features
            sku_rolling = rolling_transactions[rolling_transactions['SKUID'] == sku_id]
            feature_dict[f'SKU_RollingTotalSales_{window}Weeks'] = sku_rolling['OrderValue'].sum()
            feature_dict[f'SKU_RollingPurchaseCount_{window}Weeks'] = len(sku_rolling)
            
            # Category-specific rolling features
            category = sku_info['Category']
            category_rolling = rolling_transactions[rolling_transactions['SKUID'].isin(
                skus[skus['Category'] == category]['SKUID']
            )]
            feature_dict[f'Category_RollingTotalSales_{window}Weeks'] = category_rolling['OrderValue'].sum()
            
        # Trend-based feature: Slope of Customer_RollingPurchaseCount over weeks
        rolling_weeks = hist_transactions[
            (hist_transactions['Week'] >= week - max(ROLLING_WINDOWS)) & 
            (hist_transactions['Week'] <= week - 1) &
            (hist_transactions['CustomerID'] == customer_id)
        ]
        if not rolling_weeks.empty:
            week_counts = rolling_weeks.groupby('Week').size().reset_index(name='PurchaseCount')
            if len(week_counts) > 1:
                slope, _, _, _, _ = linregress(week_counts['Week'], week_counts['PurchaseCount'])
                feature_dict['Customer_PurchaseCount_Trend'] = slope
            else:
                feature_dict['Customer_PurchaseCount_Trend'] = 0
        else:
            feature_dict['Customer_PurchaseCount_Trend'] = 0
        
        # Granular Location Affinity (Town)
        town = customer_info['Town']
        town_transactions = hist_transactions[hist_transactions['CustomerID'].isin(
            customers[customers['Town'] == town]['CustomerID']
        )]
        for window in ROLLING_WINDOWS:
            town_window_transactions = town_transactions[
                (town_transactions['Week'] >= week - window) & 
                (town_transactions['Week'] <= week - 1)
            ]
            feature_dict[f'Town_SKU_PurchaseCount_{window}Weeks'] = len(
                town_window_transactions[town_window_transactions['SKUID'] == sku_id]
            )
        
        features_list.append(feature_dict)
    
    return pd.DataFrame(features_list)


In [ ]:
# Generate training data 
def generate_training_data(skus, customers, transactions):
    all_training_data = []
    
    for week in range(1, NUM_WEEKS + 1):
        week_transactions = transactions[transactions['Week'] == week][['Week', 'CustomerID', 'SKUID']].copy()
        week_transactions['purchase_likelihood'] = 1
        
        negative_samples = []
        for customer_id in week_transactions['CustomerID'].unique():
            purchased_skus = week_transactions[week_transactions['CustomerID'] == customer_id]['SKUID'].unique()
            non_purchased_skus = skus[~skus['SKUID'].isin(purchased_skus)]['SKUID'].sample(
                n=len(purchased_skus) * NEGATIVE_SAMPLING_RATIO
            )
            for sku_id in non_purchased_skus:
                negative_samples.append({
                    'Week': week,
                    'CustomerID': customer_id,
                    'SKUID': sku_id,
                    'purchase_likelihood': 0
                })
        
        week_data = pd.concat([
            week_transactions,
            pd.DataFrame(negative_samples)
        ], ignore_index=True)
        
        week_features = compute_features(week, transactions, skus, customers)
        week_data = week_data.merge(week_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
        
        all_training_data.append(week_data)
    
    return pd.concat(all_training_data, ignore_index=True)


In [ ]:
# Generate inference data  
def generate_inference_data(skus, customers, transactions):
    # skus, customers, transactions = generate_synthetic_data()
    
    inference_samples = []
    for customer_id in customers['CustomerID']:
        purchased_skus = transactions[transactions['CustomerID'] == customer_id]['SKUID'].unique()
        customer_state = customers[customers['CustomerID'] == customer_id]['State'].iloc[0]
        state_customers = customers[customers['State'] == customer_state]['CustomerID']
        state_transactions = transactions[transactions['CustomerID'].isin(state_customers)]
        popular_skus = state_transactions.groupby('SKUID')['OrderValue'].sum().nlargest(5).index
        customer_categories = transactions[transactions['CustomerID'] == customer_id].merge(skus, on='SKUID')['Category'].value_counts().index[:2]
        category_skus = skus[skus['Category'].isin(customer_categories)]['SKUID'].sample(5, replace=True)
        curated_skus = list(set(list(purchased_skus) + list(popular_skus) + list(category_skus)))
        
        for sku_id in curated_skus[:50]:
            inference_samples.append({
                'Week': INFERENCE_WEEK,
                'CustomerID': customer_id,
                'SKUID': sku_id
            })
    
    inference_data = pd.DataFrame(inference_samples)
    inference_features = compute_features(INFERENCE_WEEK, transactions, skus, customers, is_inference=True)
    inference_data = inference_data.merge(inference_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
    
    return inference_data

In [6]:
# skus, customers, transactions = generate_synthetic_data()
# skus.to_csv('./data/skus.csv', index=False)
# customers.to_csv('./data/customers.csv', index=False)
# transactions.to_csv('./data/transactions.csv', index=False) 

In [16]:
PATH_DIM_SKU = './data/dim_sku.csv'
PATH_DIM_CUSTOMER = './data/dim_customer.csv'
PATH_FACT_TRANSACTION = './data/fact_transaction.csv'

skus = pd.read_csv(PATH_DIM_SKU)
customers = pd.read_csv(PATH_DIM_CUSTOMER) 
transactions = pd.read_csv(PATH_FACT_TRANSACTION) 

In [19]:
transactions.head()

,Week,CustomerID,SKUID,SKUCODE,OrderCount,OrderFreqDays,OrderQty,OrderValue,lastDeliveryDate
0,13,1634267,21031,CWS012C,1,1,1,29800,30/03/2025
1,13,1634267,45899,CPL25GX120P8,1,1,2,39700,30/03/2025
2,13,1735576,22026,MIMEE_RC070,1,1,1,8000,30/03/2025
3,13,1735576,22028,MIMEE_RC100,1,1,10,103000,30/03/2025
4,13,1746988,19679,IOC001,1,1,1,10300,30/03/2025


In [21]:
datetime(2025, 1, 1)

datetime.datetime(2025, 1, 1, 0, 0)

In [20]:
timedelta(weeks = 10-1)

datetime.timedelta(days=63)

In [ ]:
# Run the data generation
if __name__ == "__main__": 
    # skus, customers, transactions = generate_synthetic_data()
    
    PATH_DIM_SKU = './data/dim_sku.csv'
    PATH_DIM_CUSTOMER = './data/dim_customer.csv'
    PATH_FACT_TRANSACTION = './data/fact_transaction.csv'
    
    skus = pd.read_csv(PATH_DIM_SKU)
    customers = pd.read_csv(PATH_DIM_CUSTOMER) 
    transactions = pd.read_csv(PATH_FACT_TRANSACTION) 
    
    
    training_data = generate_training_data(skus, customers, transactions)
    training_data.to_csv('input/training_data_multi_windows.csv', index=False)
    print("Training data with multiple rolling windows generated and saved to 'input/training_data_multi_windows.csv'")
    
    inference_data = generate_inference_data(skus, customers, transactions)
    inference_data.to_csv('input/inference_data_multi_windows.csv', index=False)
    print("Inference data with multiple rolling windows generated and saved to 'input/inference_data_multi_windows.csv'")

Training data with multiple rolling windows generated and saved to 'input/training_data_multi_windows.csv'
Inference data with multiple rolling windows generated and saved to 'input/inference_data_multi_windows.csv'


In [ ]:
print("transactions Unique Weeks: ", transactions.Week.unique().tolist())
print("training_data Unique Weeks: ", training_data.Week.unique().tolist())
print("inference_data Unique Weeks: ", inference_data.Week.unique().tolist())
print(f"{'=' * 100}")
print("transactions Unique Weeks Count: ", transactions.Week.nunique())
print("training_data Unique Weeks Count: ", training_data.Week.nunique())
print("inference_data Unique Weeks Count: ", inference_data.Week.nunique())
print(f"{'=' * 100}")
print("transactions Unique Customers: ", transactions.CustomerID.nunique())
print("training_data Unique Customers: ", training_data.CustomerID.nunique())
print("inference_data Unique Customers: ", inference_data.CustomerID.nunique())
print(f"{'=' * 100}") 
print("training_data Features: ", training_data.columns.nunique())
print("inference_data Features: ", inference_data.columns.nunique())
print(f"{'=' * 100}") 
print("training_data Features: ", training_data.drop('purchase_likelihood', axis=1).columns.tolist())
print("inference_data Features: ", inference_data.columns.tolist())

---

### **Key Changes and Improvements**

1. **Multiple Rolling Windows**:
   - The `ROLLING_WINDOWS` constant is set to `[4, 8, 12]` weeks.
   - For each rolling feature (e.g., Customer_RollingAvgOrderValue, SKU_RollingTotalSales), the code computes three versions (e.g., `_4Weeks`, `_8Weeks`, `_12Weeks`).
   - This applies to:
     - `Customer_RollingAvgOrderValue_XWeeks`
     - `Customer_RollingUniqueSKUsPurchased_XWeeks`
     - `Customer_RollingPurchaseCount_XWeeks`
     - `SKU_RollingTotalSales_XWeeks`
     - `SKU_RollingPurchaseCount_XWeeks`
     - `Category_RollingTotalSales_XWeeks`
     - `Town_SKU_PurchaseCount_XWeeks`

2. **Trend-Based Feature**:
   - Added `Customer_PurchaseCount_Trend`, which computes the slope of purchase counts over the longest window (12 weeks) using linear regression (`scipy.stats.linregress`).
   - This captures whether a customer’s purchase frequency is increasing or decreasing, adding a dynamic signal to the feature set.

3. **Sparsity Handling**:
   - For rolling features, zero values are used when no transactions exist in the window (e.g., `Customer_RollingAvgOrderValue_4Weeks = 0` if no purchases).
   - In a real-world scenario, you could add smoothing (e.g., add a small constant or use exponential moving averages) to reduce the impact of sparse data.

4. **Consistency Across Training and Inference**:
   - The same feature computation logic is used for both training and inference, ensuring the model sees consistent features.
   - Features are computed using data only from weeks prior to the target week to prevent temporal leakage.

---

### **Potential Pitfalls and Mitigations**

1. **Increased Feature Dimensionality**:
   - **Issue**: Using multiple windows increases the number of features (e.g., three versions of each rolling feature), which could lead to overfitting or computational overhead.
   - **Mitigation**: Use feature selection techniques (e.g., permutation importance, L1 regularization) during model training to identify the most predictive windows. Alternatively, reduce dimensionality using PCA or feature aggregation (e.g., mean of rolling features across windows).

2. **Sparsity in Longer Windows**:
   - **Issue**: For customers or SKUs with infrequent purchases, longer windows (e.g., 12 weeks) may result in sparse or zero-valued features.
   - **Mitigation**: Apply smoothing (e.g., add a small constant or use exponential moving averages) or aggregate features at a higher level (e.g., Category instead of SKU). The code already sets zero for empty windows, but you could enhance this with smoothing logic.

3. **Computational Cost**:
   - **Issue**: Computing features for multiple windows increases processing time, especially for large datasets.
   - **Mitigation**: Optimize with vectorized operations or use distributed frameworks like Spark/Dask for large-scale data. Cache intermediate results (e.g., rolling aggregates) to avoid redundant calculations.

4. **Feature Correlation**:
   - **Issue**: Features from different windows (e.g., Customer_RollingPurchaseCount_4Weeks and _8Weeks) may be highly correlated, reducing their unique contribution.
   - **Mitigation**: Compute correlation matrices during EDA and drop highly correlated features, or use models robust to multicollinearity (e.g., tree-based models like XGBoost).

---

### **Additional Enhancement Suggestions**

1. **Feature Aggregation Across Windows**:
   - Compute ratios or differences between windows (e.g., `Customer_RollingPurchaseCount_4Weeks / Customer_RollingPurchaseCount_12Weeks`) to capture relative changes in behavior.
   - Example: Add to `compute_features`:
     ```python
     for window in ROLLING_WINDOWS:
         if feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] > 0:
             feature_dict[f'PurchaseCount_Ratio_{window}Weeks_to_12Weeks'] = (
                 feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] /
                 feature_dict[f'Customer_RollingPurchaseCount_12Weeks'] if feature_dict[f'Customer_RollingPurchaseCount_12Weeks'] > 0 else 1
             )
         else:
             feature_dict[f'PurchaseCount_Ratio_{window}Weeks_to_12Weeks'] = 0
     ```

2. **Exponential Moving Averages**:
   - Replace simple rolling averages with exponential moving averages to give more weight to recent weeks.
   - Example: Use `pandas.ewm` for `Customer_RollingAvgOrderValue_XWeeks`.

3. **Customer Segmentation**:
   - Add features based on customer lifecycle stages (e.g., new, loyal, at-risk) by clustering customers on RFM metrics and computing rolling features per segment.

4. **Dynamic Window Selection**:
   - During training, evaluate which window sizes are most predictive using feature importance or cross-validation, and use only the top-performing windows for inference.

---

### **How to Use the Updated Code**

1. **Dependencies**:
   - Install required libraries: `pip install pandas numpy scipy`
   - The `scipy.stats.linregress` function is used for the trend-based feature.

2. **Running the Code**:
   - Save the code to a file (e.g., `sku_prediction_data_generation_multi_windows.py`).
   - Run it: `python sku_prediction_data_generation_multi_windows.py`.
   - Outputs:
     - `training_data_multi_windows.csv`: Training data with multiple rolling window features.
     - `inference_data_multi_windows.csv`: Inference data with the same features.

3. **Adapting to Real Data**:
   - Replace `generate_synthetic_data` with your actual data loading logic.
   - Extend the feature set to include all features from the project summary (e.g., Segment_RollingTotalSales_XWeeks).
   - Adjust `ROLLING_WINDOWS` or add smoothing logic based on your data characteristics.

4. **Model Training**:
   - Use the generated training data to train a model (e.g., XGBoost, LightGBM).
   - Monitor feature importance to evaluate the contribution of each window size.
   - Use the inference data to predict purchase likelihood for Week 11.

---

This updated implementation enhances the Time-Series & Rolling Window features by incorporating multiple window sizes and a trend-based feature, addressing the project’s concern about optimal window sizes and improving the model’s ability to capture diverse temporal patterns. Let me know if you’d like to explore additional enhancements, such as specific feature aggregations or smoothing techniques!